In [0]:
%python
df_taxa_bruta_nat_valid = spark.read.table("workspace.silver.taxa_bruta_natalidade")
df_esp_vida_nasc_valid = spark.read.table("workspace.silver.esperanca_vida_nascer")

In [0]:
display(df_taxa_bruta_nat_valid.limit(10))

### Aplicando regras para a camada de Validação

Regras:
- 1 - Deve ser uma das 27 siglas válidas
- 2 - Deve estar entre 1 e 12 (meses)
- 3 - Deve estar entre os 27 códigos (códigos definidos pelo IBGE para cada estado)
- 4 - Deve ser uma das 5 regiões (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)
- 5 - Deve estar entre 1 e 5 (5 regiões do Brasil)
- 6 - Deve ser uma das siglas válidas (siglas das regiões)
- 7 - Não pode ser nula porque é uma Data de referência do indicador, então considera obrigatória
- 8 - Indicador UF não deve ser nulo e deve ser maior que zero
- 9 - Indicador REG não deve ser nulo e deve ser maior que zero
- 10 - Indicador BR não deve ser nulo e deve ser maior que zero
- 11 - Validar correspondência co_uf ↔ sg_uf
- 12 - Validar coerência hierárquica regional (co_regiao_brasil deve corresponder à região da UF)
- 13 - Validar consistência dt_competencia ↔ co_ano/co_mes
- 14 - Validar range de anos razoáveis (1900 ≤ ano ≤ 2100)
- 15 - Nome da UF válido (tolera variações de caixa/acento/espaços, rejeita Inconsistência)

In [0]:
from pyspark.sql import functions as F

uf_valida = {
    "AC": 12, "AL": 27, "AM": 13, "AP": 16, "BA": 29, "CE": 23, "DF": 53,
    "ES": 32, "GO": 52, "MA": 21, "MG": 31, "MS": 50, "MT": 51, "PA": 15,
    "PB": 25, "PE": 26, "PI": 22, "PR": 41, "RJ": 33, "RN": 24, "RO": 11,
    "RR": 14, "RS": 43, "SC": 42, "SE": 28, "SP": 35, "TO": 17
}
codigo_uf_valida = list(uf_valida.values())
regiao_valida = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
sigla_regiao_valida = ["N", "NE", "SE", "S", "CO"]
mapa_regiao = {
    1: ("Norte", "N"),
    2: ("Nordeste", "NE"),
    3: ("Sudeste", "SE"),
    4: ("Sul", "S"),
    5: ("Centro-Oeste", "CO")
}
uf_regiao = {
    12: 1, 27: 2, 13: 1, 16: 1, 29: 2, 23: 2, 53: 5, 32: 3, 52: 5, 21: 2, 31: 3, 50: 5, 51: 5, 15: 1,
    25: 2, 26: 2, 22: 2, 41: 4, 33: 3, 24: 2, 11: 1, 14: 1, 43: 4, 42: 4, 28: 2, 35: 3, 17: 1
}

# Regra 15
uf_nomes = {
    11: "RONDONIA", 12: "ACRE", 13: "AMAZONAS", 14: "RORAIMA", 15: "PARA",
    16: "AMAPA", 17: "TOCANTINS", 21: "MARANHAO", 22: "PIAUI", 23: "CEARA",
    24: "RIOGRANDEDONORTE", 25: "PARAIBA", 26: "PERNAMBUCO", 27: "ALAGOAS",
    28: "SERGIPE", 29: "BAHIA", 31: "MINASGERAIS", 32: "ESPIRITOSANTO",
    33: "RIODEJANEIRO", 35: "SAOPAULO", 41: "PARANA", 42: "SANTACATARINA",
    43: "RIOGRANDEDOSUL", 50: "MATOGROSSODOSUL", 51: "MATOGROSSO",
    52: "GOIAS", 53: "DISTRITOFEDERAL"
}

regras = [
    # Regra 1
    ((~F.upper(F.col("sg_uf")).isin(*list(uf_valida.keys()))) | (F.col("sg_uf").isNull()), "UF inválida"),
    
    # Regra 2
    ((~F.col("co_mes").between(1, 12)) | (F.col("co_mes").isNull()), "Mês inválido"),
    
    # Regra 3
    ((~F.col("co_uf").isin(*codigo_uf_valida)) | (F.col("co_uf").isNull()), "Código UF inválido"),
    
    # Regra 4
    ((~F.col("no_regiao_brasil").isin(*regiao_valida)) | (F.col("no_regiao_brasil").isNull()), "Região inválida"),

    # Regra 5
    ((~F.col("co_regiao_brasil").between(1, 5)) | (F.col("co_regiao_brasil").isNull()), "Código região inválido"),
    
    # Regra 6
    ((~F.upper(F.col("sg_regiao_brasil")).isin(*sigla_regiao_valida)) | (F.col("sg_regiao_brasil").isNull()), "Sigla região inválida"),
    
    # Regra 7
    (F.col("dt_competencia").isNull(), "Data competência inválida"),
    
    # Regra 8
    ((F.col("vl_indicador_calculado_uf").isNull()) | (F.col("vl_indicador_calculado_uf") <= 0), "Indicador UF calculado inválido"),
    
    # Regra 9
    ((F.col("vl_indicador_calculado_reg").isNull()) | (F.col("vl_indicador_calculado_reg") <= 0), "Indicador REG calculado inválido"),
    
    # Regra 10
    ((F.col("vl_indicador_calculado_br").isNull()) | (F.col("vl_indicador_calculado_br") <= 0), "Indicador BR calculado inválido"),
    
    # Regra 11
    (F.col("co_uf") != F.expr(f"CASE sg_uf {' '.join([f'WHEN \"{k}\" THEN {v}' for k, v in uf_valida.items()])} END"), "Correspondência co_uf e sg_uf inválida"),
    
    # Regra 12
    (F.col("co_regiao_brasil") != F.expr(f"CASE co_uf {' '.join([f'WHEN {k} THEN {v}' for k, v in uf_regiao.items()])} END"), "Coerência regional UF → Região inválida"),
    
    # Regra 13
    ((F.year(F.to_date(F.col("dt_competencia"))) != F.col("co_ano")) | (F.month(F.to_date(F.col("dt_competencia"))) != F.col("co_mes")), "Consistência dt_competencia e co_ano/co_mes inválida"),
    
    # Regra 14
    ((F.col("co_ano") < 1900) | (F.col("co_ano") > 2100), "Ano fora do range razoável"),
    
    # Regra 15
    (  
        (F.col("no_uf").isNotNull()) & 
        (~F.expr(f"""
            array_contains(
                split(CASE co_uf {' '.join([f'WHEN {co} THEN \"{nome}|{[k for k,v in uf_valida.items() if v==co][0]}\"' for co, nome in uf_nomes.items()])} END, '[|]'),
                regexp_replace(upper(translate(no_uf, 'áàâãäéêëíóôõöúüçÁÀÂÃÄÉÊËÍÓÔÕÖÚÜÇ', 'aaaaaeeeiooooouucAAAAAEEEIOOOOOUUC')), '[ .-]', '')
            )
        """)),
        "Nome da UF inválido ou corrompido"
    )
]

# Função para aplicar as regras e gerar os motivos de rejeição
def aplicar_motivos_rejeicao(df, regras):
    return df.withColumn(
        "motivos_rejeicao",
        F.filter(
            F.array(
                *[
                    F.when(condicao, mensagem)
                    for condicao, mensagem in regras
                ]
            ),
            lambda x: x.isNotNull()
        )
    )

df_taxa_bruta_nat_valid = aplicar_motivos_rejeicao(df_taxa_bruta_nat_valid, regras)
df_esp_vida_nasc_valid = aplicar_motivos_rejeicao(df_esp_vida_nasc_valid, regras)

In [0]:
dados_teste = [
    # Registro totalmente válido
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # UF inválida (Regra 1)
    {
        "sg_uf": "XX", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Mês inválido (Regra 2)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 13, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Código UF inválido (Regra 3)
    {
        "sg_uf": "SP", "co_uf": 99, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Região inválida (Regra 4)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Leste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Código região inválido (Regra 5)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 6, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Sigla região inválida (Regra 6)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "XX", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Data competência nula (Regra 7)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": None,
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Indicador UF nulo e negativo (Regra 8)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": -1.0, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Indicador REG nulo e negativo (Regra 9)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": -1.0, "vl_indicador_calculado_br": 14.0
    },
    # Indicador BR nulo e negativo (Regra 10)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": -1.0
    },
    # Correspondência co_uf ↔ sg_uf inválida (Regra 11)
    {
        "sg_uf": "SP", "co_uf": 25, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Coerência regional UF → Região inválida (Regra 12)
    {
        "sg_uf": "PB", "co_uf": 25, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Nordeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "NE", "dt_competencia": "2025-05-01",
        "no_uf": "Paraíba",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Consistência dt_competencia ↔ co_ano/co_mes inválida (Regra 13)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 6, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Ano fora do range razoável (Regra 14)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 1800, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "1800-05-01",
        "no_uf": "São Paulo",
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Regra 15: Nome UF válido com variações aceitas (caixa/acento/espaços/hífen/sigla)
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "sao Paulo",  # Aceito: sem acento, caixa mista
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    {
        "sg_uf": "RJ", "co_uf": 33, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "RJ",  # Aceito: sigla
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    {
        "sg_uf": "RS", "co_uf": 43, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sul",
        "co_regiao_brasil": 4, "sg_regiao_brasil": "S", "dt_competencia": "2025-05-01",
        "no_uf": "Rio-Grande do Sul",  # Aceito: com hífen e espaços
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Regra 15: Nome UF inválido - prefixo corrompido
    {
        "sg_uf": "SP", "co_uf": 35, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Sudeste",
        "co_regiao_brasil": 3, "sg_regiao_brasil": "SE", "dt_competencia": "2025-05-01",
        "no_uf": "x São Paulo",  # Rejeitado: prefixo 'x' colado
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Regra 15: Nome UF inválido - erro de digitação
    {
        "sg_uf": "BA", "co_uf": 29, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Nordeste",
        "co_regiao_brasil": 2, "sg_regiao_brasil": "NE", "dt_competencia": "2025-05-01",
        "no_uf": "Bahhia",  # Rejeitado: digitação errada
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    },
    # Regra 15: Nome UF inválido - encoding corrompido
    {
        "sg_uf": "GO", "co_uf": 52, "co_mes": 5, "co_ano": 2025, "no_regiao_brasil": "Centro-Oeste",
        "co_regiao_brasil": 5, "sg_regiao_brasil": "CO", "dt_competencia": "2025-05-01",
        "no_uf": "GoiÃ¡s",  # Rejeitado: encoding corrompido (duplo encode UTF-8)
        "vl_indicador_calculado_uf": 12.5, "vl_indicador_calculado_reg": 13.0, "vl_indicador_calculado_br": 14.0
    }
]

# Converte dt_competencia de String para Date
df_teste = df_teste.withColumn("dt_competencia", F.to_date(F.col("dt_competencia")))

df_teste = spark.createDataFrame(dados_teste)
df_teste = aplicar_motivos_rejeicao(df_teste, regras)

In [0]:
display(df_teste)